# ClickHouse API (Python)

Connect, create schema, insert rows, and query — aligned with `user_activity` / `user_features` / `user_predictions`.

**Prereqs:** ClickHouse on `localhost:8123` (or `CLICKHOUSE_HOST` from Docker), `pip install -r ../requirements.txt`.

In [ ]:
import os
import sys
from pathlib import Path

SRC = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "src"
SRC = SRC.resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import clickhouse_connect
from db.clickhouse_client import ClickHouseConfig, get_client, insert_rows

cfg = ClickHouseConfig(
    host=os.environ.get("CLICKHOUSE_HOST", "localhost"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    username=os.environ.get("CLICKHOUSE_USER", "default"),
    password=os.environ.get("CLICKHOUSE_PASSWORD", ""),
    database="engagement",
)

client = get_client(cfg)
client.ping()
print("Connected:", cfg)

In [ ]:
client.command("CREATE DATABASE IF NOT EXISTS engagement")
client.database = "engagement"

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
schema_path = root / "sql" / "01_schema.sql"
sql_text = schema_path.read_text(encoding="utf-8")
for stmt in sql_text.split(";"):
    s = stmt.strip()
    if not s or s.upper().startswith("USE "):
        continue
    client.command(s)
print("Tables ready.")

In [ ]:
from datetime import datetime

demo_rows = [
    {
        "user_id": 1,
        "session_id": "s1",
        "ts": datetime(2025, 1, 10, 14, 0, 0),
        "event_type": "view",
        "product_id": 101,
        "category": "electronics",
    },
    {
        "user_id": 1,
        "session_id": "s1",
        "ts": datetime(2025, 1, 10, 14, 5, 0),
        "event_type": "click",
        "product_id": 101,
        "category": "electronics",
    },
]

insert_rows(
    client,
    "user_activity",
    demo_rows,
    ["user_id", "session_id", "ts", "event_type", "product_id", "category"],
)
client.query("SELECT count() FROM user_activity").result_rows

### Load e-shop clothing 2008 (your downloaded CSV)

Semicolon-delimited file under `clickstream+data+for+online+shopping/`. Uncomment `TRUNCATE` if you re-run after the toy demo above.

In [ ]:
from pathlib import Path
from ingestion.data_loader import load_eshop_clothing_2008_to_clickhouse

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
csv_path = root / "clickstream+data+for+online+shopping" / "e-shop clothing 2008.csv"

# client.command("TRUNCATE TABLE user_activity")
n = load_eshop_clothing_2008_to_clickhouse(csv_path, cfg=cfg)
print("Rows inserted from e-shop CSV:", n)
client.query("SELECT min(ts), max(ts), count() FROM user_activity").result_rows

In [ ]:
df = client.query_df("""
SELECT event_type, count() AS c
FROM user_activity
GROUP BY event_type
ORDER BY c DESC
""")
df

## Next steps

1. **Generic CSV:** `load_csv_to_clickhouse` if columns match `user_id`, `session_id`, `timestamp`, `event_type`, `product_id`, `category`.
2. **This dataset:** `load_eshop_clothing_2008_to_clickhouse` (see cells above).
3. Run parameterized `sql/02_feature_engineering.sql` (replace `{snapshot}`) — pick a date in range April–August 2008 from the CSV.
4. Set `high_engagement` labels in Python or `ALTER TABLE ... UPDATE` after choosing a threshold.